# Scenario 3 — Few-Shot In-Context Learning (5 / 10 / 20 examples)
## Encoder Notebook — BERT-family ICL (NO fine-tuning, NO weight updates)

**Research Question:** Can encoder models (BERT-family) perform in-context learning
comparably to decoder models (Qwen), given the same k examples and zero weight updates?

**Method:** Prompted Embedding Similarity
- Full prompt (examples + test email) fed into BERT
- [CLS] token embedding extracted — NO gradient updates
- Cosine similarity used to classify against phishing/legit prototype embeddings
- Weights frozen throughout — true ICL

**Dataset:** `darkknight25/phishing_benign_email_dataset` — strict 50/50 pool/test split

**Models:** BERT-base, RoBERTa-base, DistilBERT-base

**K values:** 5, 10, 20

**Comparison pair:** See Decoder Notebook for Qwen2.5-1.5B ICL results

In [ ]:
!nvidia-smi
!pip install -q "numpy==1.26.4" "scipy==1.12.0"
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q transformers accelerate datasets scikit-learn pandas tqdm

In [2]:
import os, re, time, warnings, random
import numpy as np, pandas as pd, torch
import torch.nn.functional as F
warnings.filterwarnings("ignore")
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from transformers import AutoTokenizer, AutoModel
from datasets import load_dataset
from tqdm.auto import tqdm

# ── Reproducibility ──────────────────────────────────────────────
SEEDS = [42, 7, 123, 0, 99]   # multiple seeds for reliable results
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {DEVICE}")
if DEVICE == "cuda": print(f"GPU    : {torch.cuda.get_device_name(0)}")

# ── Encoder models to test ───────────────────────────────────────
ENCODER_MODELS = {
    "BERT":       "bert-base-uncased",
    "RoBERTa":    "roberta-base",
    "DistilBERT": "distilbert-base-uncased",
}

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

def evaluate(y_true, y_pred, name=""):
    return {
        "Model":     name,
        "Accuracy":  f"{accuracy_score(y_true, y_pred):.4f}",
        "Precision": f"{precision_score(y_true, y_pred, zero_division=0):.4f}",
        "Recall":    f"{recall_score(y_true, y_pred, zero_division=0):.4f}",
        "F1":        f"{f1_score(y_true, y_pred, average='binary', zero_division=0):.4f}",
    }

print("Setup complete.")

Device : cuda
GPU    : Tesla T4
Setup complete.


In [3]:
# ── Dataset ──────────────────────────────────────────────────────
# STRICT split: pool and test never overlap
print("Loading dataset...")
ds = load_dataset("darkknight25/phishing_benign_email_dataset", split="train")
df = ds.to_pandas()
df["text"]  = (df["subject"].fillna("") + " " + df["body"].fillna("")).str.strip()
df["label"] = (df["label"].astype(str).str.lower() == "phishing").astype(int)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

# First 100 rows → few-shot pool only (never tested on)
# Last 100 rows → test set only (never used as examples)
pool_df = df.iloc[:100].reset_index(drop=True)
test_df  = df.iloc[100:].reset_index(drop=True)

phish_pool = pool_df[pool_df.label == 1].reset_index(drop=True)
legit_pool  = pool_df[pool_df.label == 0].reset_index(drop=True)

print(f"Pool  : {len(phish_pool)} phishing + {len(legit_pool)} legit (for examples only)")
print(f"Test  : {len(test_df)} emails (held out, never seen as examples)")
print(f"Test label distribution — Phishing: {test_df.label.sum()} | Legit: {(test_df.label==0).sum()}")

Loading dataset...


Pool  : 51 phishing + 49 legit (for examples only)
Test  : 100 emails (held out, never seen as examples)
Test label distribution — Phishing: 49 | Legit: 51


In [4]:
# ── BERT ICL via Prompted Embedding Similarity ───────────────────
#
# How it works (no fine-tuning, no weight updates):
#  1. Build k-shot prompt string for each test email
#  2. Feed full prompt into BERT → extract [CLS] embedding
#  3. Also embed k phishing examples → average = phishing prototype
#  4. Also embed k legit examples   → average = legit prototype
#  5. Classify test email by cosine similarity to each prototype
#
# This is true ICL — BERT reads examples in context, never trains.

def get_cls_embedding(model, tokenizer, text, max_len=512):
    """Extract [CLS] token embedding from BERT-family model."""
    inputs = tokenizer(
        text, return_tensors="pt",
        truncation=True, max_length=max_len,
        padding=True
    ).to(DEVICE)
    with torch.no_grad():
        outputs = model(**inputs)
    # [CLS] is always the first token
    cls_emb = outputs.last_hidden_state[:, 0, :]   # shape: (1, hidden_size)
    return F.normalize(cls_emb, dim=-1)             # L2 normalise


def build_icl_prompt(example_texts, example_labels, test_text):
    """
    Build a natural language prompt with k examples followed by
    the test email — fed as a single string into BERT.
    Format mirrors the decoder notebook exactly.
    """
    lines = ["Classify the following email as PHISHING or LEGITIMATE.\n"]
    for txt, lbl in zip(example_texts, example_labels):
        tag = "PHISHING" if lbl == 1 else "LEGITIMATE"
        lines.append(f"Email: {txt[:300]}\nLabel: {tag}\n")
    lines.append(f"Email: {test_text[:400]}\nLabel:")
    return "\n".join(lines)


def run_encoder_icl(model, tokenizer, k, phish_pool, legit_pool, test_df, seed):
    """
    Run BERT ICL for a given k and random seed.
    Returns list of predictions (0/1).
    """
    set_seed(seed)

    # Sample k//2 examples from each class
    ph_sample = phish_pool.sample(k // 2, random_state=seed)
    lg_sample  = legit_pool.sample(k // 2, random_state=seed)

    example_texts  = list(ph_sample["text"]) + list(lg_sample["text"])
    example_labels = [1] * (k // 2) + [0] * (k // 2)

    # ── Build class prototypes from example embeddings ──
    ph_embs, lg_embs = [], []
    for txt in ph_sample["text"]:
        ph_embs.append(get_cls_embedding(model, tokenizer, txt[:400]))
    for txt in lg_sample["text"]:
        lg_embs.append(get_cls_embedding(model, tokenizer, txt[:400]))

    ph_prototype = torch.stack(ph_embs).mean(0)   # average phishing embedding
    lg_prototype = torch.stack(lg_embs).mean(0)   # average legit embedding
    ph_prototype = F.normalize(ph_prototype, dim=-1)
    lg_prototype = F.normalize(lg_prototype, dim=-1)

    preds = []
    for _, row in test_df.iterrows():
        # Build prompted context (examples + test email)
        prompt = build_icl_prompt(example_texts, example_labels, row["text"])
        test_emb = get_cls_embedding(model, tokenizer, prompt)

        # Cosine similarity to each prototype
        sim_ph = F.cosine_similarity(test_emb, ph_prototype).item()
        sim_lg = F.cosine_similarity(test_emb, lg_prototype).item()

        preds.append(1 if sim_ph > sim_lg else 0)

    return preds

print("ICL functions defined.")

ICL functions defined.


In [5]:
# ── Main Experiment Loop ──────────────────────────────────────────
# For each model × each k × each seed → collect F1
# Report mean ± std across seeds (reliable, not cherry-picked)

all_results = []
K_VALUES = [5, 10, 20]

for model_name, model_id in ENCODER_MODELS.items():
    print(f"\n{'='*60}")
    print(f"Model: {model_name} ({model_id})")
    print(f"{'='*60}")

    # Load model once, freeze all weights
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model     = AutoModel.from_pretrained(model_id).to(DEVICE)
    model.eval()  # inference mode — no dropout, no gradient tracking
    for param in model.parameters():
        param.requires_grad = False   # explicitly freeze everything

    for k in K_VALUES:
        seed_f1s, seed_accs = [], []

        for seed in SEEDS:
            preds = run_encoder_icl(
                model, tokenizer, k,
                phish_pool, legit_pool, test_df, seed
            )
            f1  = f1_score(test_df.label, preds, average="binary", zero_division=0)
            acc = accuracy_score(test_df.label, preds)
            seed_f1s.append(f1)
            seed_accs.append(acc)

        mean_f1  = np.mean(seed_f1s)
        std_f1   = np.std(seed_f1s)
        mean_acc = np.mean(seed_accs)

        print(f"  k={k:>2} | F1: {mean_f1:.4f} ± {std_f1:.4f} | Acc: {mean_acc:.4f}")
        all_results.append({
            "Architecture": "Encoder",
            "Model":   f"{model_name} ICL",
            "K":       k,
            "Mean F1": f"{mean_f1:.4f}",
            "Std F1":  f"{std_f1:.4f}",
            "Mean Acc": f"{mean_acc:.4f}",
        })

    # Free GPU memory before loading next model
    del model
    torch.cuda.empty_cache()

print("\n" + "="*60)
print("SCENARIO 3 — ENCODER ICL RESULTS (mean over 5 seeds)")
print("="*60)
results_df = pd.DataFrame(all_results)
print(results_df[["K", "Model", "Mean F1", "Std F1", "Mean Acc"]].to_string(index=False))
print("\nNOTE: No fine-tuning. No weight updates. Pure in-context learning.")
print("Compare these results with the Decoder Notebook (Qwen ICL).")


Model: BERT (bert-base-uncased)


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  k= 5 | F1: 0.6517 ± 0.0143 | Acc: 0.5360
  k=10 | F1: 0.6577 ± 0.0000 | Acc: 0.4900
  k=20 | F1: 0.6577 ± 0.0000 | Acc: 0.4900

SCENARIO 3 — ENCODER ICL RESULTS (mean over 5 seeds)
 K          Model Mean F1 Std F1 Mean Acc
 5       BERT ICL  0.6855 0.0279   0.5480
10       BERT ICL  0.6595 0.0036   0.5000
20       BERT ICL  0.6603 0.0079   0.5060
 5    RoBERTa ICL  0.6457 0.0590   0.5520
10    RoBERTa ICL  0.6577 0.0000   0.4900
20    RoBERTa ICL  0.6577 0.0000   0.4900
 5 DistilBERT ICL  0.6517 0.0143   0.5360
10 DistilBERT ICL  0.6577 0.0000   0.4900
20 DistilBERT ICL  0.6577 0.0000   0.4900

NOTE: No fine-tuning. No weight updates. Pure in-context learning.
Compare these results with the Decoder Notebook (Qwen ICL).
